# Medora Phase 7 — PHI NER training

This notebook rebuilds the synthetic corpus, trains MuRIL with three seeds, exports ONNX, runs the untouched admission evaluation, and downloads the admitted bundle. No patient records are uploaded.

Before running: in Colab select **Runtime → Change runtime type → T4 GPU**. Push the corpus-generator changes to the branch configured below.

In [ ]:
import os, pathlib, subprocess, sys, torch
assert torch.cuda.is_available(), 'Enable a GPU runtime in Colab first.'
print('GPU:', torch.cuda.get_device_name(0))
REPO_URL = 'https://github.com/CSE-3200-System-Project/Medora.git'
BRANCH = 'main'  # change this if you pushed the work to another branch
WORKDIR = pathlib.Path('/content/Medora')

In [ ]:
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKDIR)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
os.chdir(WORKDIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
%pip install -q --upgrade -r tools/phi_ner/requirements-training.txt
import accelerate, huggingface_hub, transformers
print({
    'python': sys.executable,
    'transformers': transformers.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'accelerate': accelerate.__version__,
})
assert transformers.__version__ == '5.15.0'
assert int(huggingface_hub.__version__.split('.')[0]) >= 1

## Rebuild and verify the training data
The full JSONL files are reproducible build artifacts, so this creates them directly in Colab.

In [ ]:
subprocess.run([sys.executable, 'tools/phi_ner/generate_corpus.py'], check=True)
import json
manifest = json.loads(pathlib.Path('tools/phi_ner/corpus/manifest.json').read_text(encoding='utf-8'))
assert manifest['splits']['train']['rows'] + manifest['splits']['dev']['rows'] == 12000
assert manifest['pools']['source_totals']['given_name_forms'] >= 500
assert manifest['pools']['source_totals']['family_name_forms'] >= 500
assert manifest['geography_coverage']['source']['bd_admin_2022'] == 495
assert manifest['geography_coverage']['source']['bd_admin_2026_extension'] == 8
assert 60 <= manifest['frames']['phi'] + manifest['frames']['clean'] <= 100
manifest

## Train the deployable candidate
MuRIL is the primary candidate because its pretraining includes transliterated Indic text. Three seeds may take roughly 1–2 hours total on a T4. The training script selects a recall-first threshold subject to the 6% over-redaction cap.

In [ ]:
command = [
    sys.executable, 'tools/phi_ner/train.py', '--model', 'muril', '--seeds', '3',
    '--epochs', '4', '--batch-size', '16', '--max-length', '256',
    '--over-redaction-cap', '0.06'
]
print('Running:', ' '.join(command), flush=True)
with subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f'PHI training failed with exit code {return_code}; see the full error immediately above.')

## Evaluate and admit
This tests rules-only, model-only, and their union against the untouched benchmark. The command exits non-zero if the bundle misses the admission gates.

In [ ]:
subprocess.run([
    sys.executable, 'tools/phi_ner/evaluate.py', '--per-script', '--admit-bundle',
    '--bundle', 'tools/phi_ner/artifacts/deploy'
], check=True)
report = json.loads(pathlib.Path('tools/phi_ner/reports/phi_ner_eval.json').read_text(encoding='utf-8'))
report['release_gate']

In [ ]:
import shutil
archive = shutil.make_archive('/content/medora-phi-ner-muril', 'zip', 'tools/phi_ner/artifacts/deploy')
from google.colab import files
files.download(archive)

## Optional research comparisons
Run these only after saving the MuRIL bundle. XLM-R is deployable but is trained here without export so it cannot overwrite MuRIL. BanglaBERT is non-commercial and the code blocks its deployment; use it only for a research comparison.

In [ ]:
# Uncomment when you want the comparison runs.
# subprocess.run(['python', 'tools/phi_ner/train.py', '--model', 'xlmr', '--seeds', '3', '--no-export'], check=True)
# subprocess.run(['python', 'tools/phi_ner/train.py', '--model', 'banglabert', '--seeds', '3', '--allow-noncommercial', '--no-export'], check=True)